## Regression (Fatwa's popularity) for russian fatwas

In [1]:
import numpy as np
import pandas as pd
import re
import pymorphy3

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import ElasticNet, Ridge, Lasso
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression

In [2]:
df_ru = pd.read_csv('/Users/market/Desktop/thesis_project/data/df_new2_ru.csv')
#added title length as a feature
df_ru['title_length'] = df_ru['title'].str.len()
df_ru['title'] = df_ru['title'].astype(str).str.strip().str.lower()

In [ ]:
df_ru

In [3]:
# Лемматизатор
morph = pymorphy3.MorphAnalyzer()

def lemmatize_text(text):
    # Оставляем только кириллицу и пробелы, переводим в нижний регистр
    text = re.sub(r'[^а-яА-ЯёЁ\s]', ' ', text.lower())
    # Разбиваем на слова
    words = text.split()
    # Приводим каждое слово к нормальной форме
    lemmas = [morph.parse(word)[0].normal_form for word in words]
    return " ".join(lemmas)

# Применяем лемматизацию к колонке ПЕРЕД обучением (так надежнее для малых данных)
df_ru['title_cleaned'] = df_ru['title'].apply(lemmatize_text)

In [4]:
#Removing stopwords

file_path = "/Users/market/Desktop/thesis_project/stopwords/russian_stopwords.txt"

with open(file_path, "r", encoding="utf-8") as file:
    ru_stopwords = []
    for line in file:
        word = line.strip().lower()              # убрать пробелы и \n, привести к lower
        word = word.replace('"', '').replace("'", '')  # убрать кавычки
        word = re.sub(r'\s+', ' ', word)         # заменить несколько пробелов на один
        if word:                                 # пропустить пустые строки
            ru_stopwords.append(word)

In [5]:
#removing stopwords and extra spaces from titles
df_ru['title_cleaned'] = df_ru['title_cleaned'].apply(lambda x: re.sub(r'\s+', ' ', ' '.join([word for word in str(x).split() if word.lower() not in ru_stopwords])).strip())

In [144]:
df_ru['title']

0           можно ли мусульманке выйти замуж за атеиста?
1      «можно ли носить часы на левой руке? не против...
2            разрешено ли в исламе праздновать хэллоуин?
3      «раньше часто проклинал родственников, теперь ...
4      проверка на инфекционные заболевания перед бра...
                             ...                        
308            можно ли верить прогнозу погоды в исламе?
309    можно ли изгонять джиннов с помощью  растения ...
310           можно ли верить в астрологический прогноз?
311    можно ли обтирать голову при омовении  сверху ...
312    говорится ли в коране определённо точно об обя...
Name: title, Length: 313, dtype: object

## 1. Target и лог-трансформация

In [6]:
y = np.log1p(df_ru['views'])

df_ru['log_days_passed'] = np.log1p(df_ru['days_passed'])

## 2. Предикторы

In [7]:
X = df_ru[['title_cleaned', 'hijri_month', 'log_days_passed', 'title_length']]

## 3. Препроцессинг
Word-based TF-IDF 

In [8]:
word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1,2),
    min_df=3,
    max_df=0.9
)

### Числовые признаки

In [9]:
numeric_transformer = Pipeline(steps=[
    ('log', StandardScaler())
])

### One-hot кодирование исламских месяцев


In [10]:
month_encoder = OneHotEncoder(
    categories='auto',
    drop='first',
    sparse_output=True
)

## 4. ColumnTransformer 

In [11]:
preprocessor_word = ColumnTransformer(
    transformers=[
        ('word', word_vectorizer, 'title_cleaned'),
        ('month', month_encoder, ['hijri_month']),
        ('days', StandardScaler(), ['log_days_passed']),
        ('length', StandardScaler(), ['title_length'])
    ],
    remainder='drop',
    sparse_threshold=0.3
)


## 5. МОДЕЛИ

In [12]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet, HuberRegressor, BayesianRidge
from sklearn.pipeline import Pipeline

def build_models(preprocessor):
    # Шаг для конвертации в плотный массив - это нужно для моделей, которые не работают с разреженными матрицами (например, BayesianRidge и HuberRegressor)
    to_dense = FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)

    return {
        'OLS': Pipeline([
            ('preprocess', preprocessor),
            ('dense', to_dense), # Добавляем сюда
            ('model', LinearRegression())
        ]),
        
        'ElasticNet': Pipeline([
            ('preprocess', preprocessor),
            ('model', ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000, random_state=42))
        ]),
        
        'Lasso': Pipeline([
            ('preprocess', preprocessor),
            ('model', Lasso(alpha=0.01, max_iter=10000, random_state=42))
        ]),

        'BayesianRidge': Pipeline([
            ('preprocess', preprocessor),
            ('dense', to_dense), # Обязательно для этой модели
            ('model', BayesianRidge())
        ]),

        'Huber': Pipeline([
            ('preprocess', preprocessor),
            ('dense', to_dense), # Обязательно для этой модели
            ('model', HuberRegressor(max_iter=1000))
        ])
    }

In [13]:
model_groups = {
    'word_only': build_models(preprocessor_word)
}

### 6. Train-test split


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 7. Обучение + сравнение моделей

In [15]:
from sklearn.metrics import root_mean_squared_error


results = []

for group_name, models in model_groups.items():
    for model_name, pipe in models.items():

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        r2 = r2_score(y_test, y_pred)
        rmse = root_mean_squared_error(y_test, y_pred)

        results.append({
            'group': group_name,
            'model': model_name,
            'R2': r2,
            'RMSE': rmse
        })

        print(f'\n{group_name} | {model_name}')
        print(f'R² = {r2:.4f}')
        print(f'RMSE = {rmse:.4f}')


word_only | OLS
R² = 0.3668
RMSE = 0.8376

word_only | ElasticNet
R² = 0.4496
RMSE = 0.7809

word_only | Lasso
R² = 0.4349
RMSE = 0.7913

word_only | BayesianRidge
R² = 0.4475
RMSE = 0.7824

word_only | Huber
R² = 0.3192
RMSE = 0.8685


## Сравнение моделей

In [16]:
results_df = pd.DataFrame(results)
results_df.sort_values(['group', 'R2'], ascending=[True, False])

,group,model,R2,RMSE
1,word_only,ElasticNet,0.449578,0.780933
3,word_only,BayesianRidge,0.447503,0.782403
2,word_only,Lasso,0.434904,0.791274
0,word_only,OLS,0.366808,0.837594
4,word_only,Huber,0.319208,0.868506


In [17]:
results_df.pivot_table(
    index='group',
    columns='model',
    values='R2'
)

model,BayesianRidge,ElasticNet,Huber,Lasso,OLS
group,,,,,
word_only,0.447503,0.449578,0.319208,0.434904,0.366808


## 9. Извлечение коэффициентов

In [178]:
# 1. Берем конкретную модель
pipeline = model_groups['word_only']['ElasticNet']

# 2. Обучаем её (без этого коэффициенты .coef_ физически не существуют в памяти)
pipeline.fit(X_train, y_train)

# 3. Теперь достаем данные
feature_names = pipeline.named_steps['preprocess'].get_feature_names_out()
coefs = pipeline.named_steps['model'].coef_.ravel()

# 4. Собираем таблицу
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': coefs
})

# Очистка имен
coef_df['feature'] = coef_df['feature'].str.replace(r'^.*__', '', regex=True)

# Сортировка
coef_df = coef_df.sort_values('coef', ascending=False)

## 10. Топ-20 слов и символных паттернов, повышающих просмотры


In [179]:
# Создаем список "технических" префиксов, которые надо убрать
exclude_prefixes = [ 'log_days_passed', 'hijri_month', 'title_length']

# Фильтруем: оставляем только те признаки, которые НЕ содержат эти слова
# case=False на случай разного регистра
clean_coef_df = coef_df[~coef_df['feature'].str.contains('|'.join(exclude_prefixes), case=False)]

# Выводим чистый топ
print(clean_coef_df.head(20))

        feature      coef
8           дуа  0.646358
2         волос  0.480141
33         пост  0.456080
25         мясо  0.444798
9       женщина  0.357607
6         джинн  0.258089
46         рыба  0.245441
34  праздновать  0.237280
0         аборт  0.208597
24  мусульманка  0.196725
30     омовение  0.195009
27        намаз  0.171389
28       носить  0.156780
42          рай  0.000000
43      рамадан  0.000000
44      ребёнок  0.000000
45         рука  0.000000
41  разрешаться -0.000000
51        сунна -0.000000
47        слово  0.000000


### 12. Отдельно интерпретируем контролли


In [180]:
coef_df[coef_df['feature'].str.contains('month')].sort_values('coef', ascending=False)


,feature,coef
67,hijri_month_Sha'ban,0.568513
65,hijri_month_Ramadan,0.325169
66,hijri_month_Safar,0.103133
63,hijri_month_Rabi al-Thani,0.100801
60,hijri_month_Jumada al-Thani,-0.000000
61,hijri_month_Muharram,-0.000000
62,hijri_month_Rabi al-Awwal,0.000000
68,hijri_month_Shawwal,-0.000000
64,hijri_month_Rajab,-0.089187
59,hijri_month_Jumada al-Awwal,-0.118653


In [181]:
coef_df[coef_df['feature'].str.contains('title_length')]

,feature,coef
70,title_length,-0.068773


In [182]:
coef_df[coef_df['feature'].str.contains('log_days_passed')]

,feature,coef
69,log_days_passed,0.508148
